**Author:** Salvador Navas  
**Date:** 2025-06-27

### GPM 3IMERG

**GPM 3IMERG (Global Precipitation Measurement Integrated Multi-satellite Retrievals for GPM)**
is a high-resolution global precipitation product produced by NASA.

- **Spatial resolution:** ~0.1° (~10 km)
- **Temporal resolution:** Half-hourly, daily, monthly
- **Temporal coverage:** 1998–present
- **Spatial coverage:** 60°S–60°N

> This notebook uses **synthetic demo data** that reproduces the GPM format
> without requiring NASA Earthdata credentials.
> Replace the synthetic section with actual `GPMDownloader()` calls when credentials are available.

In [1]:
# GPMDownloader requires 'earthaccess' + NASA Earthdata credentials
# Interactive map requires 'ipyleaflet' — both skipped in this demo
# from pyhydra.data_sources.rainfall import GPMDownloader
# from pyhydra.utils import interactive_map

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import xarray as xr
import tempfile, os
print('Imports OK')

Imports OK


---
## GPMDownloader — real usage

```python
from pyhydra.data_sources.rainfall import GPMDownloader

gpm = GPMDownloader()  # authenticates via earthaccess.login()
gpm.set_region(points=[(39.39, -0.62)])  # Valencia
results = gpm.search('2020-01-01', '2020-01-31', resolution='daily')
data = gpm.open_dataset(results, variable='precipitationCal', output_folder='gpm_data')
```

In [2]:
# === SYNTHETIC DEMO DATA ===
# Creates an xarray Dataset matching GPM IMERG format
# Variables: precipitationCal  Coords: lat, lon, time

TMPDIR = tempfile.mkdtemp()
rng = np.random.default_rng(42)

times = pd.date_range('2020-01-01', '2020-12-31', freq='D')
lats  = np.arange(37.0, 44.1, 0.1)   # Iberian Peninsula
lons  = np.arange(-9.0, 4.1,  0.1)

# Synthetic precipitation: gamma-distributed values
shape_arr = rng.gamma(0.5, 3.0, (len(times), len(lats), len(lons))).astype('float32')
# Add seasonal peak in autumn for Iberia
doy = times.dayofyear.values[:, None, None]
seasonal_factor = 1 + 0.8 * np.sin(2 * np.pi * (doy - 270) / 365)
precip = (shape_arr * seasonal_factor).astype('float32')

ds_gpm = xr.Dataset(
    {'precipitationCal': (['time', 'lat', 'lon'], precip)},
    coords={
        'time': times,
        'lat':  lats,
        'lon':  lons,
    }
)
ds_gpm['precipitationCal'].attrs.update({
    'long_name': 'GPM IMERG Final precipitation estimate (gauge-calibrated)',
    'units': 'mm/hr',
})

NC_FILE = os.path.join(TMPDIR, 'gpm_demo_2020.nc')
ds_gpm.to_netcdf(NC_FILE)
print(f'Synthetic GPM NetCDF created: {NC_FILE}')
print(ds_gpm)

Synthetic GPM NetCDF created: /var/folders/44/dw7p6q9108xcc4mmh_f7q1vc0000gn/T/tmpj_joulp9/gpm_demo_2020.nc
<xarray.Dataset> Size: 14MB
Dimensions:           (time: 366, lat: 72, lon: 131)
Coordinates:
  * time              (time) datetime64[ns] 3kB 2020-01-01 ... 2020-12-31
  * lat               (lat) float64 576B 37.0 37.1 37.2 37.3 ... 43.9 44.0 44.1
  * lon               (lon) float64 1kB -9.0 -8.9 -8.8 -8.7 ... 3.7 3.8 3.9 4.0
Data variables:
    precipitationCal  (time, lat, lon) float32 14MB 4.338 6.903 ... 0.06986


---
## 1. Extract point time series

In [3]:
# Extract precipitation at Valencia (39.39°N, -0.62°E)
LAT, LON = 39.39, -0.62

ds = xr.open_dataset(NC_FILE)
ts = ds['precipitationCal'].sel(lat=LAT, lon=LON, method='nearest')
df_gpm = ts.to_dataframe().reset_index()[['time', 'precipitationCal']]
df_gpm = df_gpm.rename(columns={'time': 'date', 'precipitationCal': 'precip_mm_hr'})
df_gpm = df_gpm.set_index('date').sort_index()
print(df_gpm.describe())
df_gpm.head()

       precip_mm_hr
count    366.000000
mean       1.382588
std        2.365031
min        0.000004
25%        0.098597
50%        0.517562
75%        1.707890
max       19.682949


,precip_mm_hr
date,
2020-01-01,0.269927
2020-01-02,0.151810
2020-01-03,1.726838
2020-01-04,2.570044
2020-01-05,19.323748


---
## 2. Spatial mean and monthly aggregation

In [4]:
# Area mean over Iberian Peninsula
area_mean = ds['precipitationCal'].mean(dim=['lat', 'lon'])
df_area = area_mean.to_dataframe().rename(columns={'precipitationCal': 'precip_area_mean'})
df_area.index.name = 'date'

# Daily total converted to mm/day (hourly rate * 24)
df_area['precip_mm_day'] = df_area['precip_area_mean'] * 24
monthly = df_area['precip_mm_day'].resample('ME').sum()

print('Monthly accumulated precipitation (mm/month):')
print(monthly.round(1))

Monthly accumulated precipitation (mm/month):
date
2020-01-31    1944.699951
2020-02-29    1585.699951
2020-03-31    1287.800049
2020-04-30     806.500000
2020-05-31     450.399994
2020-06-30     241.300003
2020-07-31     281.799988
2020-08-31     542.000000
2020-09-30     927.500000
2020-10-31    1415.199951
2020-11-30    1725.199951
2020-12-31    1990.500000
Freq: ME, Name: precip_mm_day, dtype: float32


---
## 3. Visualisation

In [5]:
fig, axes = plt.subplots(2, 1, figsize=(14, 7))

axes[0].plot(df_area.index, df_area['precip_mm_day'], lw=0.7, color='navy', alpha=0.6)
axes[0].set_ylabel('Precipitation (mm/day)')
axes[0].set_title('GPM IMERG — Area mean over Iberian Peninsula (synthetic demo)', fontsize=13)
axes[0].grid(alpha=0.3)

axes[1].bar(monthly.index.month, monthly.values, color='royalblue', alpha=0.85,
            width=0.7)
axes[1].set_xticks(range(1, 13))
axes[1].set_xticklabels(['Jan','Feb','Mar','Apr','May','Jun',
                          'Jul','Aug','Sep','Oct','Nov','Dec'])
axes[1].set_ylabel('Monthly precipitation (mm)')
axes[1].set_title('Monthly accumulation — 2020')
axes[1].grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('/tmp/GPM_download.png', dpi=100, bbox_inches='tight')
plt.close()
print('Plot saved to /tmp/GPM_download.png')
ds.close()

Plot saved to /tmp/GPM_download.png
